In [ ]:
# --- Tutorial setup ---
# This cell loads everything needed to run this notebook standalone.
# It is hidden in the rendered documentation but visible when running the
# notebook as a tutorial.
import sys
sys.path.insert(0, "/".join(["..", "python"]))

import warnings

import matplotlib.pyplot as plt
import numpy as np
from notebook_utils import mpl_plot_wrapper

from gridr.core.convolution.fft_filtering import fft_array_filter
from gridr.core.filter.antialiasing.reciprocal_cell import (
    ReciprocalCellFilter,
    kernel_response,
)
from gridr.core.filter.lattice import voronoi_cell_vertices
from gridr.core.grid.grid_antialiasing import resolve_antialiasing_filter
from gridr.core.grid.grid_resampling import array_grid_resampling
from gridr.misc.mandrill import mandrill

# print the warnings of GridR without the file and line they come from
warnings.formatwarning = lambda msg, cat, *args, **kwargs: f"{cat.__name__}: {msg}\n"
plt.rcParams.update({"figure.dpi": 100, "font.size": 9, "axes.titlesize": 9})

@mpl_plot_wrapper
def show_images(images, titles, scale=1, prefix=None):
    """Show images side by side, each image pixel on ``scale x scale`` screen pixels.

    An integer scale and nearest-neighbor display keep the screen from
    resampling the images, which would add its own aliasing.
    """
    dpi = plt.rcParams["figure.dpi"]
    gap, top = 12, 22
    sizes = [(im.shape[1] * scale, im.shape[0] * scale) for im in images]
    width = sum(w for w, _ in sizes) + gap * (len(images) + 1)
    height = max(h for _, h in sizes) + top + gap
    fig = plt.figure(figsize=(width / dpi, height / dpi), dpi=dpi)
    x = gap
    for im, title, (w, h) in zip(images, titles, sizes, strict=True):
        ax = fig.add_axes((x / width, gap / height, w / width, h / height))
        ax.imshow(im, cmap="gray", vmin=0, vmax=1, interpolation="nearest")
        ax.set_title(title)
        ax.axis("off")
        x += w + gap
    return fig


@mpl_plot_wrapper
def plot_kernel(kernel_filter, facets, tolerance, prefix=None):
    """Plot the center of a kernel, its 2D response and its response at f_y = 0."""
    kernel = kernel_filter.kernel
    f, response = kernel_response(kernel)
    c = kernel.shape[0] // 2
    r = min(15, c)
    center = kernel[c - r:c + r + 1, c - r:c + r + 1]
    vmax = np.abs(kernel).max()
    cell = voronoi_cell_vertices(facets)
    cell = np.vstack([cell, cell[:1]])
    # cell boundary on the f_x axis: |f_x a_x| = 1/2 for the closest facet
    edge = min(0.5 / abs(a[0]) for a in facets if abs(a[0]) > 1e-12)

    fig, axs = plt.subplots(1, 3, figsize=(10, 3.2))
    im = axs[0].imshow(center, cmap="RdBu_r", vmin=-vmax, vmax=vmax,
                       extent=(-r - 0.5, r + 0.5, r + 0.5, -r - 0.5))
    axs[0].set(title=f"kernel {kernel.shape[0]} x {kernel.shape[1]}, center",
               xlabel="column", ylabel="row")
    fig.colorbar(im, ax=axs[0], shrink=0.8)

    im = axs[1].imshow(response, cmap="viridis", vmin=0, vmax=1, origin="lower",
                       extent=(f[0], f[-1], f[0], f[-1]))
    axs[1].plot(*cell.T, ls=":", color="w")
    axs[1].set(xlim=(f[0], f[-1]), ylim=(f[0], f[-1]), title="response",
               xlabel="$f_x$ (cycles/pixel)", ylabel="$f_y$ (cycles/pixel)")
    fig.colorbar(im, ax=axs[1], shrink=0.8)

    axs[2].plot(f, response[len(f) // 2])
    axs[2].axvline(edge, ls=":", color="0.5")
    axs[2].axhline(tolerance, ls=":", color="0.5")
    axs[2].set(xlim=(0, 0.5), title="response at $f_y = 0$",
               xlabel="$f_x$ (cycles/pixel)")
    fig.tight_layout()
    return fig


@mpl_plot_wrapper
def plot_axis_responses(kernel_filters, labels, title, prefix=None):
    """Plot the responses of kernels along f_x, at f_y = 0."""
    fig, ax = plt.subplots(figsize=(7, 2.8))
    for kernel_filter, label in zip(kernel_filters, labels, strict=True):
        f, response = kernel_response(kernel_filter.kernel)
        ax.plot(f, response[len(f) // 2], label=label)
    ax.set(xlim=(0, 0.5), ylim=(-0.05, 1.1), title=title,
           xlabel="$f_x$ (cycles/pixel), at $f_y = 0$", ylabel="kernel response")
    ax.legend(loc="lower left")
    fig.tight_layout()
    return fig





# Antialiasing filter for a decimation by 4

This tutorial builds the antialiasing filter of a resampling that reduces the resolution by 4 on both axes, applies
it, and shows the effect of each parameter. The reasons for the filter and the method are given in the theoretical
foundations: {doc}`/theoretical_foundations/antialiasing/aliasing_and_reciprocal_lattice` for aliasing,
{doc}`/theoretical_foundations/antialiasing/source_prefiltering` for the filter.

**What you'll learn**

- How to build the filter of a grid with `ReciprocalCellFilter` and `resolve_antialiasing_filter`
- How to apply it to the source before the resampling
- What each parameter changes, and when to change it
- What the two experimental options do

## Setting things up

We import the specification of the filter, `ReciprocalCellFilter`, from `gridr.core.filter.antialiasing.reciprocal_cell`,
and `resolve_antialiasing_filter` from `gridr.core.grid.grid_antialiasing`, which builds the filter of a grid.

In [ ]:
from gridr.core.filter.antialiasing.reciprocal_cell import (
    ReciprocalCellFilter,
    kernel_response,
)
from gridr.core.grid.grid_antialiasing import resolve_antialiasing_filter

We then import the GridR functions that filter and resample an image, `fft_array_filter` and `array_grid_resampling`,
and wrap them in two short helpers: `prefilter` convolves an image with the kernel of a filter, `resample` resamples
it on a grid with the cubic interpolator.

In [ ]:
from gridr.core.convolution.fft_filtering import fft_array_filter
from gridr.core.grid.grid_resampling import array_grid_resampling

def resample(img, grid_row, grid_col):
    """Resample an image on a full-resolution grid with the cubic interpolator."""
    out, _ = array_grid_resampling(
        "cubic", np.ascontiguousarray(img), grid_row, grid_col, (1, 1), None,
        boundary_condition="edge", trust_padding=True,
    )
    return out

def prefilter(img, kernel_filter):
    """Convolve an image with the kernel of a filter, mirroring the borders."""
    return fft_array_filter(img, kernel_filter.kernel, boundary="reflect").data

## Test images

We use two images of 512 x 512 pixels. The zone plate is a map of the frequency plane: its local frequency grows
linearly from the center, up to 0.45 cycle per pixel at the edge of the disk, where the rings fade to gray over the
last 16 pixels of radius ({ref}`Figure 1 <aa-fig-zp_source>`). The mandrill, first channel, shows the same effect on a natural image.

In [ ]:
from gridr.misc.mandrill import mandrill
import numpy as np

N = 512       # size of the test images
FMAX = 0.45   # local frequency of the zone plate at radius N / 2, in cycles per pixel
TAPER = 16    # width, in pixels, over which the rings fade out at the edge of the disk


def zone_plate(n=N, fmax=FMAX):
    """Zone plate whose local frequency grows linearly to ``fmax`` at radius n/2."""
    y, x = np.mgrid[0:n, 0:n].astype(float)
    r2 = (x - n / 2) ** 2 + (y - n / 2) ** 2
    k = fmax / (n / 2)
    t = np.clip((n / 2 - np.sqrt(r2)) / TAPER, 0.0, 1.0)
    w = 0.5 - 0.5 * np.cos(np.pi * t)
    return 0.5 + 0.5 * w * np.cos(np.pi * k * r2)

ZONE_PLATE = zone_plate()
MANDRILL = mandrill[0].astype(np.float64) / 255.0

In [ ]:
_ = show_images(
    [ZONE_PLATE],
    ["zone plate, one image pixel per screen pixel"],
    prefix="antialiasing_001_sources_zone_plate",
)

In [ ]:
_ = show_images(
    [MANDRILL],
    ["mandrill"],
    prefix="antialiasing_001_sources_mandrill",
)

## The grid

A decimation by 4 is a backward grid whose destination pixel $(i, j)$ reads the source at row $4i + 1.5$ and column
$4j + 1.5$: the center of the block of 4 x 4 source pixels it covers. The destination has 128 x 128 pixels.

In [ ]:
i, j = np.mgrid[0:128, 0:128].astype(np.float64)
grid_row = 4.0 * i + 1.5
grid_col = 4.0 * j + 1.5

## Without filter

In [ ]:
_ = show_images(
    [resample(ZONE_PLATE, grid_row, grid_col),
     resample(MANDRILL, grid_row, grid_col)],
    ["zone plate, resampled no filter", "mandrill, resampled no filter"],
    scale=2,
    prefix="antialiasing_001_resampled_no_filter",
)

The destination shows false rings all over the zone plate, and a grainy texture on the fur of the mandrill. The
destination can only represent the source frequencies below $1/8$ cycle per pixel on each axis; every finer detail
folds back into a false, coarser pattern ({ref}`aa-sec-aliasing`). The interpolator does not prevent it.

## Building the filter

The filter is described by a specification, `ReciprocalCellFilter`, whose fields are the parameters of the method.
`resolve_antialiasing_filter` estimates the transition matrix $W$ of the grid, the source displacement for one
destination column and one destination row, and builds the kernel for it. It returns the filter, a `KernelFilter`,
and diagnostics.

In [ ]:
spec = ReciprocalCellFilter()
kernel_filter, diag = resolve_antialiasing_filter(spec, grid_row, grid_col)

print(f"W =\n{diag.transition_matrix}")
print(f"kernel: {kernel_filter.kernel.shape}")
print(f"accuracy met: {diag.filter.accuracy_met}")
print(f"response error: {diag.filter.response_error:.4f}")

$W = 4I$: the grid steps by 4 source pixels per destination pixel on both axes. The filter keeps the frequencies of
the square $\lvert f_x\rvert, \lvert f_y\rvert \le 1/8$, the cell of the reciprocal lattice of $W$, and removes the
others with a smooth transition ({ref}`aa-sec-template`). The kernel is the smallest odd size whose response follows
this template within `kernel_accuracy` ({ref}`aa-sec-synthesis`).

The kernel and its response, computed by `kernel_response`. The dotted lines mark the cell and the tolerance.

In [ ]:
_ = plot_kernel(
    kernel_filter, diag.filter.facets, spec.aliasing_tolerance, prefix="antialiasing_001_kernel"
)

The kernel is the product of two 1D kernels, one per axis, because the template is a product of gates on $f_x$ and
$f_y$. Its central lobe is about 9 pixels wide, a little more than twice the decimation. The response is 1 inside the cell, falls to the tolerance 0.1 on the boundary $f = 1/8$, and to
almost 0 beyond.

### When $W$ is known

`resolve_antialiasing_filter` estimates $W$ on the grid and calls the method `design` of the specification. When
$W$ is known, as for a decimation, `design` builds the filter directly, without a grid. $W = [\mathbf w_1\
\mathbf w_2]$ holds the source displacement for one destination column, $\mathbf w_1$, and for one destination row,
$\mathbf w_2$, each written (column, row). `design` returns the diagnostics of the method directly, without the
estimation of $W$.

In [ ]:
kernel_filter_w, diag_w = ReciprocalCellFilter().design(4.0 * np.eye(2))
print(f"same kernel: {np.array_equal(kernel_filter_w.kernel, kernel_filter.kernel)}")

## Applying the filter

The kernel is applied to the source by convolution, here with `fft_array_filter`, and the filtered source is then
resampled as usual.

In [ ]:
zone_plate_filtered = prefilter(ZONE_PLATE, kernel_filter)
mandrill_filtered = prefilter(MANDRILL, kernel_filter)

In [ ]:
_ = show_images(
    [resample(zone_plate_filtered, grid_row, grid_col),
     resample(mandrill_filtered, grid_row, grid_col)],
    ["zone plate, resampled filtered", "mandrill, resampled filtered"],
    scale=2,
    prefix="antialiasing_001_resampled_filtered",
)

Only the central rings of the zone plate remain, those below $1/8$ cycle per pixel; the rest becomes a uniform gray,
as it should. The mandrill loses the grain of its fur. The convolution reads a margin around the source, given by
`kernel_filter.margins()`: a tiled processing must read each tile with this margin.

In [ ]:
print(f"margins (top, bottom, left, right): {kernel_filter.margins()}")

## The parameters

### `aliasing_tolerance`: aliasing against blur

The tolerance $\varepsilon$ is the value of the template on the cell boundary: a frequency just beyond the boundary
folds with an amplitude at most $\varepsilon$ ({ref}`aa-gate-and-tolerance`). A smaller value moves the transition
inside the cell: less aliasing, more blur. The default 0.1 removes most of the aliasing for a small loss of
sharpness; 0.5 centers the transition on the boundary; 0.01 suits measurement uses.

In [ ]:
images, titles = [], []
for tolerance in (0.5, 0.1, 0.01):
    spec = ReciprocalCellFilter(aliasing_tolerance=tolerance)
    kf, _ = resolve_antialiasing_filter(spec, grid_row, grid_col)
    images.append(resample(prefilter(ZONE_PLATE, kf), grid_row, grid_col))
    titles.append(f"aliasing_tolerance = {tolerance}")
_ = show_images(images, titles, scale=2, prefix="antialiasing_001_tolerance")

At 0.5, faint false rings remain around the central pattern, in the corners of the square. At 0.01 they are gone, and
the last rings of the pattern are a little weaker.

### `cutoff_tanh_slope` and `kernel_accuracy`: the kernel size

The slope sets the sharpness of the transition, and `kernel_accuracy` the error allowed between the response of the
kernel and the template. A sharper transition or a smaller error needs a larger kernel ({ref}`aa-sec-size`). The size
grows in proportion to the slope, but only slowly with the accuracy.

In [ ]:
print("slope  accuracy  kernel size")
for slope in (5.0, 10.0, 20.0):
    for accuracy in (0.01, 0.001):
        spec = ReciprocalCellFilter(cutoff_tanh_slope=slope, kernel_accuracy=accuracy)
        kf, _ = resolve_antialiasing_filter(spec, grid_row, grid_col)
        print(f"{slope:5g}  {accuracy:8g}  {kf.kernel.shape[0]:11d}")

The defaults, slope 10 and accuracy 0.01, give a 77 x 77 kernel here. The convolution goes through the FFT, so a
larger kernel costs little computation, but it widens the margins.

### `max_kernel_size` and `kernel_size`: limiting the size

By default the size is bounded by twice its estimate, which a decimation by 4 never reaches. `max_kernel_size` sets
an explicit bound: if the accurate kernel is larger, it is truncated to the bound, the diagnostics report that the
accuracy is not met, and a warning gives the size that would meet it ({ref}`aa-sec-sizebound`). `kernel_size` forces
a size, without warning.

In [ ]:
for options in ({"max_kernel_size": 41}, {"kernel_size": 41}):
    spec = ReciprocalCellFilter(**options)
    kf, d = resolve_antialiasing_filter(spec, grid_row, grid_col)
    print(f"{options}: kernel {kf.kernel.shape}, accuracy met: {d.filter.accuracy_met}")
    print(f"    response error: {d.filter.response_error:.3f}, "
          f"accurate size: {d.filter.accurate_size}")

### `isotropic_frame`: the shape of the cell

This parameter selects the metric in which the cell is computed ({ref}`aa-sec-cell`). It only matters when the
columns of $W$ are not orthogonal, for a sheared grid. For a decimation along the axes, every metric gives the same
square cell, and the default `"destination"` is the right choice.

## Experimental options

The two options are off by default and have only been assessed on synthetic grids
({doc}`/theoretical_foundations/antialiasing/experimental_options`). A decimation by 4 on both axes needs neither:
the options change nothing for it. Each one is shown on the grid it is meant for.

### `experimental_nyquist_gates`: oblique grids near unit scale

On a rotation at unit scale, the cell overflows the Nyquist square of the source, and the template jumps on its
border. No finite kernel follows the jump: the accuracy is not met, and the warning points to this option
({ref}`aa-sec-periodicity`). The option adds gates on the Nyquist square where the template jumps.

In [ ]:
angle = np.deg2rad(30.0)
r, c = np.mgrid[0:512, 0:512].astype(np.float64) - 255.5
rotation_row = 255.5 + np.sin(angle) * c + np.cos(angle) * r
rotation_col = 255.5 + np.cos(angle) * c - np.sin(angle) * r

resampled_gates = {}
kernel_gates = {}
for nyquist_gates in (False, True):
    spec = ReciprocalCellFilter(experimental_nyquist_gates=nyquist_gates)
    kf, d = resolve_antialiasing_filter(spec, rotation_row, rotation_col)
    kernel_gates[nyquist_gates] = (kf, d)
    print(f"experimental_nyquist_gates={nyquist_gates}: kernel {kf.kernel.shape}")
    print(f"    accuracy met: {d.filter.accuracy_met}, "
          f"border jump: {d.filter.border_jump:.2f}")

    zone_plate_filtered = prefilter(ZONE_PLATE, kernel_filter)
    mandrill_filtered = prefilter(MANDRILL, kernel_filter)
    resampled_gates[nyquist_gates] = [
        resample(zone_plate_filtered, grid_row, grid_col),
        resample(mandrill_filtered, grid_row, grid_col)
    ]

In [ ]:
_ = plot_kernel(
    kernel_gates[False][0], kernel_gates[False][1].filter.facets, spec.aliasing_tolerance, prefix="antialiasing_001_kernel_no_nyquist_gate"
)

In [ ]:
_ = show_images(
    resampled_gates[False],
    ["zone plate, resampled no nyquist_gates", "mandrill, resampled no nyquist_gates"],
    scale=2,
    prefix="antialiasing_001_resampled_no_nyquist_gates",
)

In [ ]:
_ = plot_kernel(
    kernel_gates[True][0], kernel_gates[True][1].filter.facets, spec.aliasing_tolerance, prefix="antialiasing_001_kernel_no_nyquist_gate"
)

In [ ]:
_ = show_images(
    resampled_gates[True],
    ["zone plate, resampled nyquist_gates", "mandrill, resampled nyquist_gates"],
    scale=2,
    prefix="antialiasing_001_resampled_nyquist_gates",
)

With the option, the jump falls below the tolerance and the accuracy is met. The cost is a little blur near the
border of the Nyquist square.

### `experimental_progressive_gates`: an axis kept

When the rows are decimated by 4 and the columns kept, nothing folds along the columns. Yet by default the gate of
that axis still attenuates its high frequencies. The option drops or softens the gate of a facet that folds little or
nothing ({ref}`aa-sec-active`).

In [ ]:
i, j = np.mgrid[0:128, 0:512].astype(np.float64)
rows_row, rows_col = 4.0 * i + 1.5, j

kernel_filters = []
for progressive in (False, True):
    spec = ReciprocalCellFilter(experimental_progressive_gates=progressive)
    kf, d = resolve_antialiasing_filter(spec, rows_row, rows_col)
    gates = np.round(d.filter.template_tolerances, 3)
    print(f"experimental_progressive_gates={progressive}: gate values {gates}")
    kernel_filters.append(kf)

In [ ]:
_ = plot_axis_responses(
    kernel_filters,
    ["default", "progressive gates"],
    "rows decimated by 4, columns kept",
    prefix="antialiasing_001_progressive",
)

By default the response along the kept axis falls to the tolerance at the source Nyquist frequency, although nothing
folds along that axis. With the option the gate of the columns is dropped, and the response stays at 1 up to the
Nyquist frequency.

## Summary

For a simple decimation, the defaults are a good start:

```python
spec = ReciprocalCellFilter()
kernel_filter, diag = resolve_antialiasing_filter(spec, grid_row, grid_col)
kernel = kernel_filter.kernel
filtered = fft_array_filter(source, kernel, boundary="reflect").data
```

then resample `filtered` with the grid. `kernel_filter` is `None` when no filter is needed, for a zoom for instance.

| Parameter | Default | Change it when |
|---|---|---|
| `aliasing_tolerance` | 0.1 | 0.01 for measurements, 0.5 for more sharpness |
| `cutoff_tanh_slope` | 10.0 | the kernel must be smaller, or sharper |
| `kernel_accuracy` | 0.01 | rarely |
| `max_kernel_size` | `None` | the margins must stay small |
| `kernel_size` | `None` | a fixed size is required |
| `isotropic_frame` | `"destination"` | sheared grids only |
| `experimental_nyquist_gates` | `False` | the warning suggests it |
| `experimental_progressive_gates` | `False` | grids that keep or zoom an axis |